# BERT 文本分类

这个 Notebook 展示 `BERT（Bidirectional Encoder Representations from Transformers）` 在 `AG News` 新闻分类任务上的完整 fine-tune 流程。

内容包括：
- AG News 数据集加载与预处理
- Tokenizer 编码与 DataLoader 构建
- BERT Encoder-Only 架构解读
- Token 形状变化分析（input_ids → hidden states → logits）
- 与 GPT-2 的编码器 vs 解码器对比
- 训练、验证与预测展示
- 参数量统计

## 1. 环境准备

```bash
pip install torch transformers datasets
```

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertForSequenceClassification, BertTokenizer
from datasets import load_dataset

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    model_name: str = 'bert-base-uncased'
    # AG News 有 4 个类别：World、Sports、Business、Sci/Tech
    num_classes: int = 4
    max_length: int = 128
    batch_size: int = 32
    num_workers: int = 2
    lr: float = 2e-5
    # BERT fine-tune 通常 3~5 个 epoch 就足够收敛
    epochs: int = 3
    # 控制每个 split 实际使用的样本量，便于快速实验
    train_samples: int = 4000
    test_samples: int = 800

cfg = Config()
cfg

## 2. 加载 AG News 数据集

AG News 是常用的英文新闻分类基准，共 4 类，训练集 12 万条，测试集 7600 条。

| 标签 | 类别 |
|------|------|
| 0 | World |
| 1 | Sports |
| 2 | Business |
| 3 | Sci/Tech |

In [ ]:
raw = load_dataset('ag_news')
train_raw = raw['train'].select(range(cfg.train_samples))
test_raw = raw['test'].select(range(cfg.test_samples))

class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

print(f'训练集：{len(train_raw)} 条')
print(f'测试集：{len(test_raw)} 条')
print(f'示例：{train_raw[0]}')

In [ ]:
# 各类别分布，确认数据均衡
import collections
counter = collections.Counter(train_raw['label'])
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([class_names[i] for i in sorted(counter)], [counter[i] for i in sorted(counter)])
ax.set_title('AG News 训练集类别分布')
ax.set_ylabel('样本数')
plt.tight_layout()
plt.show()

## 3. Tokenizer 与 DataLoader

BERT 使用 WordPiece 分词，输入格式固定为：

```
[CLS] token1 token2 ... tokenN [SEP]
```

- `[CLS]`：Classification Token，最终用它的 hidden state 做分类
- `[SEP]`：句子分隔符，单句任务也需要

In [ ]:
tokenizer = BertTokenizer.from_pretrained(cfg.model_name)

# 查看分词效果
sample = train_raw[0]['text']
tokens = tokenizer(sample, max_length=cfg.max_length, truncation=True)
print('原文：', sample[:80], '...')
print('Token IDs：', tokens['input_ids'][:20], '...')
print('Token 数量：', len(tokens['input_ids']))

In [ ]:
class AGNewsDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # padding 到统一长度，使同 batch 张量形状一致
        enc = self.tokenizer(
            item['text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }


train_dataset = AGNewsDataset(train_raw, tokenizer, cfg.max_length)
test_dataset = AGNewsDataset(test_raw, tokenizer, cfg.max_length)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

batch = next(iter(train_loader))
print('input_ids shape :', batch['input_ids'].shape)
print('attention_mask  :', batch['attention_mask'].shape)
print('label           :', batch['label'].shape)

## 4. BERT 结构解读

### 4.1 Encoder-Only vs GPT-2

| 维度 | BERT | GPT-2 |
|------|------|-------|
| 架构 | Encoder-Only | Decoder-Only |
| 注意力 | 双向（看全文） | 单向（只看左侧） |
| 预训练任务 | MLM + NSP | 自回归语言建模 |
| 适合任务 | 理解（分类、NER、问答） | 生成（续写、摘要） |

### 4.2 BERT 预训练机制

- **MLM（Masked Language Modeling）**：随机遮住 15% 的 token，让模型预测被遮住的词。双向注意力使模型能同时利用左右上下文。
- **NSP（Next Sentence Prediction）**：判断两句话是否相邻，让模型学习句间关系。

### 4.3 CLS Token 分类

Fine-tune 时，取 `[CLS]` 位置的 hidden state（维度 768），接一个线性层输出类别 logits。

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    cfg.model_name,
    num_labels=cfg.num_classes,
)
model = model.to(device)
print(model.config)

## 5. Token 尺寸变化分析

输入 → Embedding → 12 层 Transformer Encoder → CLS hidden state → 线性分类头

In [ ]:
@torch.no_grad()
def inspect_shapes(model, batch):
    input_ids = batch['input_ids'][:1].to(device)
    attention_mask = batch['attention_mask'][:1].to(device)

    print(f'input_ids          -> {tuple(input_ids.shape)}   (batch, seq_len)')

    # 只取 encoder 输出（不含分类头）
    outputs = model.bert(input_ids=input_ids, attention_mask=attention_mask)
    last_hidden = outputs.last_hidden_state
    cls_hidden = last_hidden[:, 0, :]

    print(f'last_hidden_state  -> {tuple(last_hidden.shape)}  (batch, seq_len, hidden_dim)')
    print(f'CLS hidden state   -> {tuple(cls_hidden.shape)}   (batch, hidden_dim)')

    logits = model.classifier(cls_hidden)
    print(f'logits             -> {tuple(logits.shape)}       (batch, num_classes)')


inspect_shapes(model, batch)

## 6. 参数量统计

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total = count_parameters(model)
print(f'BERT-base 可训练参数：{total:,}（约 {total/1e6:.1f}M）')

## 7. 训练与验证函数

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# AdamW 相比 Adam 正确地将权重衰减从自适应学习率中解耦
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)

total_steps = len(train_loader) * cfg.epochs
# warmup 防止 fine-tune 初期梯度过大破坏预训练权重
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)


def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        # 梯度裁剪防止预训练权重被过大梯度冲垮
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        preds = outputs.logits.argmax(dim=-1)
        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        preds = outputs.logits.argmax(dim=-1)
        total_loss += outputs.loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

## 8. 训练主循环

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(cfg.epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc = evaluate(model, test_loader, device)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch+1}/{cfg.epochs}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  '
          f'val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

In [ ]:
epochs_range = range(1, cfg.epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='train loss')
axes[0].plot(epochs_range, history['val_loss'], label='val loss')
axes[0].set_title('Loss 曲线')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], label='train acc')
axes[1].plot(epochs_range, history['val_acc'], label='val acc')
axes[1].set_title('Accuracy 曲线')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. 预测结果展示

In [ ]:
@torch.no_grad()
def show_predictions(model, dataloader, class_names, device, n=8):
    model.eval()
    batch = next(iter(dataloader))
    input_ids = batch['input_ids'][:n].to(device)
    attention_mask = batch['attention_mask'][:n].to(device)
    labels = batch['label'][:n]

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    preds = logits.argmax(dim=-1).cpu()

    for i in range(n):
        text = tokenizer.decode(input_ids[i].cpu(), skip_special_tokens=True)[:80]
        gt = class_names[labels[i]]
        pred = class_names[preds[i]]
        mark = '✓' if gt == pred else '✗'
        print(f'[{mark}] GT={gt:8s}  Pred={pred:8s}  | {text}...')


show_predictions(model, test_loader, class_names, device)